# Week 4: from messy data to model-ready (stress outcome)

This notebook works through the 12-step preprocessing workflow on `dataset.csv`: 5,000 students, their social-media habits, sleep, study time, `Stress_Level` and a `Mental_Health_Score`.

**Research question:** Is daily social media use associated with self-reported stress levels among young adults?

**Goal:** prepare the data to predict `Stress_Level` (Low < Medium < High < Very High). There is no model yet; that comes in Week 5.

**How to run it:** select the `.venv` kernel, then click **Run All**. The cells must run top to bottom, because each one uses variables from the ones before it.

## Setup
Load the libraries, then the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

In [ ]:
# dataset.csv lives in the data/ folder next to this notebook.
DATA_PATH = "data/dataset.csv"

students = pd.read_csv(DATA_PATH)
students.head()

## B1. Inspect
Three checks: shape and types, then missing values, duplicates and summary statistics, then category counts.

In [ ]:
print("Rows, columns:", students.shape)
students.info()

In [ ]:
print("Missing values:", int(students.isna().sum().sum()))
print("Exact duplicate rows:", students.duplicated().sum())
students.describe().round(2)

In [ ]:
st_categorical = ["Gender", "Country", "Academic_Level", "Most_Used_Platform",
                  "Purpose_Of_Use", "Stress_Level"]
for col in st_categorical:
    counts = students[col].value_counts(dropna=False)
    print(f"--- {col} ({counts.size} categories) ---")
    print(counts.head(8), "\n")

**What the output shows**

- **5,000 rows × 13 columns** with **no missing values**. The dataset is fairly clean.
- **2 exact duplicate rows.**
- **Read the min and max:** `Physical_Activity_Hours` has a **minimum of −0.4**, which is impossible because nobody exercises for negative hours. Everything else is plausible: ages 18–24, sleep 3.6–9.9 h.
- `Country` has **over 100 categories**, many with only a handful of students. This matters for encoding in B9.
- The labels are already consistent (`Male`/`Female`, three academic levels).

## B2. Row-wise cleaning (before the split)
1. Drop exact duplicates.
2. Strip stray spaces from text labels, as a safety rule. There are no spelling variants here.
3. Set **impossible** values to `NaN`: negative hours, or more than 24 hours in a day.

In [ ]:
st_clean = students.drop_duplicates().copy()

for col in st_categorical:
    st_clean[col] = st_clean[col].str.strip()

hour_cols = ["Avg_Daily_Usage_Hours", "Study_Hours",
             "Physical_Activity_Hours", "Sleep_Hours_Per_Night"]
for col in hour_cols:
    impossible = (st_clean[col] < 0) | (st_clean[col] > 24)
    print(f"{col}: {impossible.sum()} impossible values set to NaN")
    st_clean.loc[impossible, col] = np.nan

# Sanity check across columns: total hours per day can't exceed 24.
total_hours = st_clean[hour_cols].sum(axis=1)
print("\nMaximum total hours in one day:", total_hours.max())
print("Rows after removing duplicates:", len(st_clean))

In [ ]:
# Checkpoint
assert st_clean.duplicated().sum() == 0, "Duplicates remain"
assert (st_clean[hour_cols].dropna() >= 0).all().all(), "Negative hours remain"
assert (st_clean[hour_cols].dropna() <= 24).all().all(), "More than 24 hours remain"
print("B2 checks passed:", st_clean.shape)

## B2b. Rare platforms and usage bands
1. **Rare platforms:** platforms with fewer than 100 users are grouped as `Other`, so tiny groups don't produce unreliable comparisons.
2. **Usage bands:** daily social-media hours grouped into under 3, 3–5, 5–7 and 7+ hours, for comparing stress across light and heavy users. The band is for descriptive comparisons only; models use the continuous hours.

In [ ]:
platform_counts = st_clean["Most_Used_Platform"].value_counts()
rare_platforms = platform_counts[platform_counts < 100].index.tolist()
print("Platforms grouped as Other:", rare_platforms)
print("Rows affected:", int(st_clean["Most_Used_Platform"].isin(rare_platforms).sum()))
st_clean["Most_Used_Platform"] = st_clean["Most_Used_Platform"].replace(rare_platforms, "Other")

st_clean["Usage_Band"] = pd.cut(st_clean["Avg_Daily_Usage_Hours"], bins=[0, 3, 5, 7, np.inf],
                                 labels=["<3h", "3-5h", "5-7h", "7h+"], right=False)
print("\nStudents per usage band:")
print(st_clean["Usage_Band"].value_counts().sort_index())

## B3. Target and features — check for target leakage

Target: `Stress_Level`, encoded in order as Low = 1, Medium = 2, High = 3, Very High = 4.

**Think:** is `Mental_Health_Score` a *cause* we could know in advance, or **another measurement of the same outcome**, recorded at the same moment?

**Decision:** we **drop `Mental_Health_Score`** from the features. It's self-reported in the same questionnaire as stress and measures closely related wellbeing (average score is 7.8 for *Low* stress and 5.1 for *Very High*). If we want to explain stress from **behaviour**, mental-health score is part of the outcome, not an input. A model using it would mostly be learning "stressed people report poorer wellbeing".

*This is a judgement call. If mental health were measured earlier, for example at the start of term, it could be a legitimate feature.*

There's no ID column to remove.

In [ ]:
st_target = "Stress_Level"
st_leaky = ["Mental_Health_Score"]
st_stress_order = ["Low", "Medium", "High", "Very High"]

# Evidence for the leakage decision: mental-health score at each stress level
print(students.groupby("Stress_Level")["Mental_Health_Score"].mean().reindex(st_stress_order).round(2))

st_X = st_clean.drop(columns=[st_target, "Usage_Band"] + st_leaky)   # band is for comparisons only
st_y = st_clean[st_target].map({s: i for i, s in enumerate(st_stress_order, start=1)})   # Low=1 ... Very High=4
assert st_y.notna().all(), "Unexpected Stress_Level label"

print("\nFeatures used:", st_X.columns.tolist())
print("\nStress distribution:")
print(st_clean[st_target].value_counts().reindex(st_stress_order))

## B3b. Initial findings: social media use and stress
Descriptive look at the research question on the cleaned data, before any modelling. Spearman correlation suits stress because it is ordinal.

In [ ]:
st_habits = ["Avg_Daily_Usage_Hours", "Daily_Unlocks", "Sleep_Hours_Per_Night",
             "Study_Hours", "Physical_Activity_Hours"]

# 1. Mean habits at each stress level
print(st_clean.groupby(st_target)[st_habits].mean().reindex(st_stress_order).round(2))

# 2. Spearman correlation of each habit with stress (0-3)
print("\nSpearman correlation with stress:")
print(st_clean[st_habits + ["Age"]].corrwith(st_y, method="spearman").round(2).sort_values())

# 3. Share of each stress level within each usage band (%)
print("\nStress level by usage band (% of band):")
print((pd.crosstab(st_clean["Usage_Band"], st_clean[st_target], normalize="index")[st_stress_order] * 100).round(1))

# 4. Is the pattern consistent across academic level? (mean usage hours)
print("\nMean usage hours by academic level and stress:")
print(st_clean.groupby(["Academic_Level", st_target])["Avg_Daily_Usage_Hours"]
      .mean().unstack()[st_stress_order].round(2))

# 5. Chart: usage hours by stress level
ordered = pd.Categorical(st_clean[st_target], st_stress_order, ordered=True)
st_clean.assign(**{st_target: ordered}).boxplot(column="Avg_Daily_Usage_Hours", by=st_target)
plt.suptitle("")
plt.title("Daily social media hours by stress level")
plt.ylabel("Hours per day")
plt.tight_layout()
plt.show()

## B4. Train / test split

In [ ]:
# stratify keeps the same share of each stress level in train and test (only 13% are Low)
st_X_train, st_X_test, st_y_train, st_y_test = train_test_split(
    st_X, st_y, test_size=0.2, random_state=42, stratify=st_y
)
print("train:", st_X_train.shape, " test:", st_X_test.shape)

# Working copies for the manual steps B5–B10; the originals are kept for the pipeline in B11.
sXtr, sXte = st_X_train.copy(), st_X_test.copy()

## B5. Missing values
The only gaps are the ones **we created** in B2 (negative activity hours). Physical activity is a skewed count of hours, so we use the **training median**.

In [ ]:
st_numeric = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks", "Study_Hours",
              "Physical_Activity_Hours", "Sleep_Hours_Per_Night"]

print("Missing in train before:")
print(sXtr.isna().sum()[sXtr.isna().sum() > 0])

st_medians = sXtr[st_numeric].median()          # learned from TRAIN only
sXtr[st_numeric] = sXtr[st_numeric].fillna(st_medians)
sXte[st_numeric] = sXte[st_numeric].fillna(st_medians)

print("\nMissing after:", int(sXtr.isna().sum().sum()), "train /", int(sXte.isna().sum().sum()), "test")

## B6. Outliers
We use the IQR rule (values beyond 1.5 × IQR from the quartiles) to *flag* unusual values. Flagged values are investigated, not automatically deleted.

In [ ]:
q1, q3 = sXtr[st_numeric].quantile(0.25), sXtr[st_numeric].quantile(0.75)
iqr = q3 - q1
flagged = ((sXtr[st_numeric] < q1 - 1.5 * iqr) | (sXtr[st_numeric] > q3 + 1.5 * iqr)).sum()
print("Values flagged by the IQR rule (train):")
print(flagged)

sXtr[["Avg_Daily_Usage_Hours", "Daily_Unlocks", "Sleep_Hours_Per_Night"]].hist(bins=25, figsize=(12, 3), layout=(1, 3))
plt.tight_layout()
plt.show()

**Answer:** Only a few values are flagged, and all of them are plausible: about 9 hours of social media a day, or 270 phone unlocks. These are heavy users, not errors, and they're exactly the students whose wellbeing we care most about. **We keep them.**

## B7. Dates
`dataset.csv` has **no date column**, so this step doesn't apply. In real projects, check whether "no date" means the data really is a single snapshot. If it is, the model can't learn anything about change over time.

## B8. Feature engineering
Three new features, created from the original units before encoding and scaling:

1. **`social_to_study_ratio`** — social-media hours per study hour. The *balance* between the two may matter more than either one on its own.
2. **`unlocks_per_usage_hour`** — how often the phone is picked up per hour of use. Many short, compulsive checks are different from a few long sessions.
3. **`sleep_deficit`** — hours below the recommended 7 hours of sleep (0 if 7 or more). This captures the idea that lack of sleep hurts, while extra sleep beyond 7 hours doesn't keep helping.

In [ ]:
def st_add_features(data):
    result = data.copy()
    result["social_to_study_ratio"] = (
        result["Avg_Daily_Usage_Hours"] / result["Study_Hours"].clip(lower=0.1)
    )
    result["unlocks_per_usage_hour"] = (
        result["Daily_Unlocks"] / result["Avg_Daily_Usage_Hours"].clip(lower=0.1)
    )
    result["sleep_deficit"] = (7 - result["Sleep_Hours_Per_Night"]).clip(lower=0)
    return result

sXtr = st_add_features(sXtr)
sXte = st_add_features(sXte)

st_new = ["social_to_study_ratio", "unlocks_per_usage_hour", "sleep_deficit"]
# Quick check: how does each new feature relate to stress in TRAIN? (Spearman, as stress is ordinal)
print(sXtr[st_new].corrwith(st_y_train, method="spearman").round(2))
sXtr[st_new].head()

## B9. Encoding
| Column | Type | Encoder |
|---|---|---|
| `Academic_Level` | ordinal (High School < Undergraduate < Graduate) | `OrdinalEncoder`, explicit order |
| `Gender`, `Most_Used_Platform`, `Purpose_Of_Use` | nominal | `OneHotEncoder` |
| `Country` | nominal with **100+ categories** | `OneHotEncoder(min_frequency=20)`: rare countries are grouped into one *infrequent* column |

**Why `min_frequency`?** Without it, `Country` alone would create over 100 columns, most of them almost always 0. A country seen only twice in training can't teach the model anything reliable.

In [ ]:
st_level_order = ["High School", "Undergraduate", "Graduate"]
st_ordinal_enc = OrdinalEncoder(categories=[st_level_order])
sXtr[["Academic_Level"]] = st_ordinal_enc.fit_transform(sXtr[["Academic_Level"]])
sXte[["Academic_Level"]] = st_ordinal_enc.transform(sXte[["Academic_Level"]])

st_nominal = ["Gender", "Country", "Most_Used_Platform", "Purpose_Of_Use"]
st_onehot = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                          sparse_output=False)
tr_nom = pd.DataFrame(st_onehot.fit_transform(sXtr[st_nominal]),
                      columns=st_onehot.get_feature_names_out(st_nominal), index=sXtr.index)
te_nom = pd.DataFrame(st_onehot.transform(sXte[st_nominal]),
                      columns=st_onehot.get_feature_names_out(st_nominal), index=sXte.index)

sXtr = pd.concat([sXtr.drop(columns=st_nominal), tr_nom], axis=1)
sXte = pd.concat([sXte.drop(columns=st_nominal), te_nom], axis=1)

print("Columns after encoding:", sXtr.shape[1])
print("Country columns:", [c for c in tr_nom.columns if c.startswith("Country")])

## B10. Scaling

In [ ]:
st_scale_cols = st_numeric + st_new
st_scaler = StandardScaler()
sXtr[st_scale_cols] = st_scaler.fit_transform(sXtr[st_scale_cols])   # learn on train
sXte[st_scale_cols] = st_scaler.transform(sXte[st_scale_cols])       # reuse on test

print("Train means (should be ~0):", sXtr[st_scale_cols].mean().round(2).abs().max())
print("Test means (close to 0, but not exactly):")
print(sXte[st_scale_cols].mean().round(2))

## B11. The whole thing as one pipeline
Row-wise features first, then a `ColumnTransformer` that sends each group of columns to its own mini-pipeline. The imputers stay in the pipeline even though this dataset has almost no gaps, because **new data might**.

In [ ]:
st_numeric_all = st_numeric + st_new

st_preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), st_numeric_all),
    ("ord", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OrdinalEncoder(categories=[st_level_order]))]), ["Academic_Level"]),
    ("nom", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                               min_frequency=20, sparse_output=False))]), st_nominal),
])

st_pipeline = Pipeline([
    ("features", FunctionTransformer(st_add_features)),
    ("preprocess", st_preprocessor),
])

st_train_prepared = st_pipeline.fit_transform(st_X_train)   # learn from train
st_test_prepared = st_pipeline.transform(st_X_test)         # reuse on test

print("Prepared train:", st_train_prepared.shape, "| test:", st_test_prepared.shape)
print("Missing values left:", int(np.isnan(st_train_prepared).sum()))
print("Same result as the manual steps:", st_train_prepared.shape[1] == sXtr.shape[1])

## B12. Leakage, revisited

In [ ]:
col = "Daily_Unlocks"
print(f"mean from ALL data   : {st_X[col].mean():.2f}")
print(f"mean from TRAIN only : {st_X_train[col].mean():.2f}")
print(f"SD   from ALL data   : {st_X[col].std():.2f}")
print(f"SD   from TRAIN only : {st_X_train[col].std():.2f}")

**Answer**

- **Preprocessing leakage:** the differences are tiny, but the scaler and imputer still learn from **train only**, so the test set remains a fair stand-in for new students.
- **Target leakage:** the risky column here is `Mental_Health_Score`, not an obvious "after the fact" column such as a cancellation date. It is recorded at the same time and is strongly tied to stress, so it is really part of the outcome. A random split wouldn't flag it: the model would score very well, and the score would be misleading if the aim is to explain stress from behaviour.

### Summary of decisions
| Step | Decision |
|---|---|
| Duplicates | 2 removed |
| Impossible values | negative `Physical_Activity_Hours` → `NaN` → train median |
| Target | `Stress_Level`, encoded Low=1 to Very High=4; stratified split |
| Target leakage | `Mental_Health_Score` dropped (co-measured outcome) |
| Rare platforms | platforms with <100 users grouped as `Other` |
| Usage bands | <3, 3–5, 5–7, 7+ h (descriptive comparisons only) |
| Outliers | flagged, investigated, **kept** (plausible heavy users) |
| Dates | none in this dataset |
| New features | `social_to_study_ratio`, `unlocks_per_usage_hour`, `sleep_deficit` |
| Encoding | ordinal `Academic_Level`; one-hot for the rest, rare countries grouped |
| Scaling | `StandardScaler`, fitted on train |